1) Load dataset

In [71]:
import pandas as pd


In [72]:
df_path = "../data/processed/feature_engineered_with_SLM.csv"
df = pd.read_csv(df_path)

df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 33 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   transaction_id             1200 non-null   int64  
 1   customer_id                1200 non-null   int64  
 2   transaction_datetime       1200 non-null   object 
 3   country                    1200 non-null   object 
 4   channel                    1200 non-null   object 
 5   merchant_category          1200 non-null   object 
 6   amount                     1200 non-null   float64
 7   device_trust_score         1200 non-null   float64
 8   num_txn_24h_customer       1200 non-null   int64  
 9   previous_chargeback_count  1200 non-null   int64  
 10  is_fraud                   1200 non-null   int64  
 11  transaction_note           1200 non-null   object 
 12  slm_risk_score             1200 non-null   int64  
 13  slm_urgency                1200 non-null   objec

In [73]:
df.head()

,transaction_id,customer_id,transaction_datetime,country,channel,merchant_category,amount,device_trust_score,num_txn_24h_customer,previous_chargeback_count,...,txn_hour_count,txn_dayofweek_count,customer_mean_amount,customer_std_amount,amount_z_customer,is_risky_country,low_device_trust,high_amount_z,composite_risk_score,urgency_numeric
0,998,1000,2022-10-05 15:13:00,US,mobile_app,luxury,672.47,23.0,2,3,...,1,2,1931.283333,1662.07937,-0.757373,0,1,0,7.333333,2
1,953,1000,2023-02-05 06:18:18,US,web,gaming,2593.31,84.0,0,0,...,1,2,1931.283333,1662.07937,0.398312,0,0,0,3.555556,2
2,94,1000,2023-03-23 18:04:02,US,pos_terminal,electronics,302.22,70.0,3,1,...,1,2,1931.283333,1662.07937,-0.980136,0,0,0,4.000000,2
3,457,1000,2023-04-05 00:14:27,US,mobile_app,gaming,4395.07,24.0,0,2,...,1,2,1931.283333,1662.07937,1.482352,0,1,0,7.333333,2
4,1199,1000,2023-07-13 02:02:46,US,mobile_app,luxury,579.99,23.0,1,0,...,1,2,1931.283333,1662.07937,-0.813014,0,1,0,5.555556,0


2. Define target & features

In [74]:
target_col = 'is_fraud'   
drop_cols = ['transaction_id', 'customer_id', 'transaction_datetime', 
             'prev_txn_time', 'transaction_note', 'txn_7d_count', 'is_fraud']

x = df.drop(columns=drop_cols)
y = df[target_col].astype(int)


In [75]:
# Identify numerical and categorical columns
categorical_cols = [col for col in x.columns if x[col].dtype == 'object']
numeric_cols = [col for col in x.columns if col not in categorical_cols]

print("Categorical features:", categorical_cols)
print("Numeric features:", numeric_cols)

Categorical features: ['country', 'channel', 'merchant_category', 'slm_urgency', 'slm_anomaly_category']
Numeric features: ['amount', 'device_trust_score', 'num_txn_24h_customer', 'previous_chargeback_count', 'slm_risk_score', 'txn_hour', 'txn_dayofweek', 'txn_week', 'is_weekend', 'is_night', 'txn_gap_minutes', 'txn_hour_count', 'txn_dayofweek_count', 'customer_mean_amount', 'customer_std_amount', 'amount_z_customer', 'is_risky_country', 'low_device_trust', 'high_amount_z', 'composite_risk_score', 'urgency_numeric']


3. Train-test split

In [76]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    x, y, test_size=0.3, stratify=y, random_state=42
)

4. Preprocess data 

In [77]:
# Simple imputation for both, one-hot encoding for categorical data and scaling for numerical data
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

num_tf = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_tf = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(drop='first', handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", num_tf, numeric_cols),
        ("cat", categorical_tf, categorical_cols)
    ], remainder="drop")


